# Baseline CNN

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from keras.utils import set_random_seed

SEED = 42
IMG_SIZE = 224
NUM_CLASSES = 11
CACHE = f'../raw_data/crops_{IMG_SIZE}.npz'

set_random_seed(SEED)

# Load the crops (only train and validation are used here, not the test set)
data = np.load(CACHE)
X_train, y_train = data['X_train'], data['y_train']
print(X_train.shape, y_train.shape)

## The validation split, grouped by image

In [ ]:
train_df = train_df.reset_index(drop=True)

# Step 1: one row per image, with its class
image_table = train_df.groupby('image_id')['category_id'].first().reset_index()

# Step 2: stratified split on images
train_ids, val_ids = train_test_split(
    image_table['image_id'],
    test_size=0.2,
    stratify=image_table['category_id'],
    random_state=SEED,
)

# Step 3: map back to the crops
is_val = train_df['image_id'].isin(val_ids).to_numpy()

X_tr, y_tr = X_train[~is_val], y_train[~is_val]
X_val, y_val = X_train[is_val], y_train[is_val]
print(X_tr.shape, X_val.shape)

In [ ]:
train_df[['egg_id', 'image_id']].assign(split=np.where(is_val, 'val', 'train')).to_csv('../split.csv', index=False)

# TEST

In [ ]:
"""from sklearn.model_selection import train_test_split

# Step 1: class of each crop, read from the one-hot labels
labels = y_train.argmax(axis=1)

# Step 2: stratified split on crop positions
indices = np.arange(len(X_train))
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=SEED,
)

# Step 3: pick the crops
X_tr, y_tr = X_train[train_idx], y_train[train_idx]
X_val, y_val = X_train[val_idx], y_train[val_idx]
print(X_tr.shape, X_val.shape)"""

## Build the CNN

In [ ]:
from keras import Input
from keras.models import Sequential
from keras.layers import Rescaling, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_cnn(img_size=IMG_SIZE, num_classes=NUM_CLASSES):
    model = Sequential()
    model.add(Input(shape=(img_size, img_size, 3)))
    model.add(Rescaling(1.0 / 255))        # your crops are uint8 (0-255)

    # Block 1
    model.add(Conv2D(32, kernel_size=3, padding='same', activation='relu'))
    model.add(MaxPooling2D(pool_size=2))

    # Block 2
    model.add(Conv2D(64, kernel_size=3, padding='same', activation='relu'))
    model.add(MaxPooling2D(pool_size=2))

    # Block 3
    model.add(Conv2D(128, kernel_size=3, padding='same', activation='relu'))
    model.add(MaxPooling2D(pool_size=2))

    # Classifier
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(num_classes, activation='softmax'))

    return model

model = build_cnn()
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',   # because y is one-hot
    metrics=['accuracy'],
)
model.summary()

### Train with early stopping

In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint

os.makedirs('../models', exist_ok=True)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,                   # stop after 5 epochs with no improvement
    restore_best_weights=True,    # go back to the best epoch
)
checkpoint = ModelCheckpoint(
    '../models/baseline_cnn.keras',
    monitor='val_loss',
    save_best_only=True,
)

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    callbacks=[early_stopping, checkpoint],
)

### Learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.show()

### Record baseline score

In [ ]:
y_prob = model.predict(X_val)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.argmax(y_val, axis=1)

print(classification_report(y_true, y_pred))
print('Macro F1:', round(f1_score(y_true, y_pred, average='macro'), 3))

### Current "preprocessing" check

In [ ]:
import os
CACHE = '../raw_data/crops_224.npz'
print(os.path.exists(CACHE))

In [ ]:
import numpy as np

data = np.load(CACHE)
X_train, y_train = data['X_train'], data['y_train']
print(X_train.shape, X_train.dtype, y_train.shape)

In [ ]:
fake = np.random.randint(0, 256, size=(2, 224, 224, 3)).astype('uint8')
output = model.predict(fake)
print(output.shape)          # (2, 11)
print(output.sum(axis=1))    # each row about 1.0

In [ ]:
import matplotlib.pyplot as plt

class_names = [
    "Ascaris", "Capillaria", "Enterobius", "Fasciolopsis", "Hookworm",
    "H. diminuta", "H. nana", "Opisthorchis", "Paragonimus", "Taenia", "Trichuris",
]

indices = np.random.choice(len(X_train), size=9, replace=False)

fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax, i in zip(axes.ravel(), indices):
    ax.imshow(X_train[i])
    ax.set_title(class_names[np.argmax(y_train[i])])
    ax.axis('off')
plt.show()

In [ ]:
small_X = X_train[:200]
small_y = y_train[:200]

history_small = model.fit(small_X, small_y, epochs=30, batch_size=32, verbose=1)
print(history_small.history['accuracy'][-1])